# Now testing with England/UK

Have worked out how to plot H3 population density with pydeck in `pydeck_test.ipynb` and have arrived at a few different funcitones, shown below

In [ ]:
import pydeck as pdk
import pandas as pd
import numpy as np
import h3
import io
import requests
import geodatasets as gds
import geopandas as gpd
import numpy as np
import pycountry
from geopy.geocoders import Nominatim
from pathlib import Path

In [ ]:
def pop_density_calc(df):
    # First should probs check that it is the right dataframe input
    if df.columns.to_list()[0:2] != ["h3", "population"]:
        raise ValueError("Nu uh uh, wrong dataframe initial two columns name, should be ['h3', 'population', ...]")
    
    # To be honest, probs wont trip the above, but only problems will be if there's an alpha channel already or a population density already produced
    # Should probs add more error tripping
    
    area_km_calc_format = lambda x: h3.cell_area(x,"km^2")
    df["population/km^2"] = df["population"] / (df["h3"].map(area_km_calc_format))
    return df

In [ ]:
def to_parent_aggregate_3(df, required_res):
    if required_res >= 8:
        raise ValueError("why are u doing that. The maximum resolution data we have is 8 so this is silly. Choose < 8")
    df_temp = df.copy(deep=True)

    # Check if all h3 res 8 cells
    get_res = lambda x: h3.get_resolution(x)
    res = pd.Series(data=df_temp["h3"].map(get_res),name="res")
    
    if not res.eq(8).all():
        raise ValueError(f"Wrong resolution for one/some input H3 hexes, supposed to be res. 8")
    
    to_parent_format = lambda x: h3.cell_to_parent(x, required_res)
    df_temp["h3"] = df["h3"].map(to_parent_format)
    df_temp = df_temp.groupby("h3")["population"].sum().reset_index()
    df_temp = pop_density_calc(df_temp)
    return df_temp

In [ ]:
def get_country_coords(country_name):
    geolocator = Nominatim(user_agent="GeospatialCancerAccess2")
    try:
        location = geolocator.geocode(country_name)
        if location:
            return (location.latitude, location.longitude)
        else:
            return None
    except Exception as e:
        print(f"Error: {e}")
        return None

In [ ]:
def pop_density_log_opacity_plot(
        df, 
        country,
        min_pop_dens=1, 
        max_pop_dens=50000,
        h3_column="h3",
        pickable=True,
        stroked=False,
        filled=True,
        extruded=False,
        get_elevation="population/km^2",
        elevation_scale=20,
        high_precision=True,
        auto_highlight=True,
        pitch=30,
        bearing=0,
        map_provider="carto",
        map_style="light", # road, dark, satellite, dark_no_labels, light_no_labels,
        map_output_folder="H3_pydeck_maps",
        map_name="log_opacity_map",
        open_browser=True,
        ):

    lp = np.log(df["population/km^2"].clip(lower=min_pop_dens, upper=max_pop_dens))
    lp_min, lp_max = np.log(min_pop_dens), np.log(max_pop_dens)

    z = (lp - lp_min) / (lp_max - lp_min) # in [0, 1]

    df_temp= df.copy(deep=True)

    df_temp["alpha"] = (30 + z * (255 - 30)).round().astype(int)
    # Purely for formatting and displaying in the tooltip
    df_temp["population_dens_2dp"] = df_temp["population/km^2"].map(lambda x: round(x,2))

    layer = pdk.Layer(
        "H3HexagonLayer",
        df_temp,
        pickable=pickable,
        stroked=stroked,
        filled=filled,
        extruded=extruded,
        get_elevation=get_elevation,
        elevation_scale=elevation_scale,
        high_precision=high_precision,
        get_hexagon=h3_column,
        auto_highlight=auto_highlight,
        get_fill_color="[0, 122, 255, alpha]",  # data-driven transparency
        get_line_color=[255, 255, 255],
        line_width_min_pixels=1,
    )

    country_name = pycountry.countries.lookup(country).name
    latitude, longitude = get_country_coords(country_name)

    view_state = pdk.ViewState(latitude=latitude, longitude=longitude, zoom=7, bearing=bearing, pitch=pitch)

    tooltip = {
        "html": "Population: {population},<br/>Population density: {population_dens_2dp}/km^2",
        "style": {
            "backgroundColor": "steelblue",
            "color": "white"
        }
    }


    r = pdk.Deck(layers=[layer], 
                initial_view_state=view_state,
                map_provider=map_provider,
                map_style=map_style,  # road, dark, satellite, dark_no_labels, light_no_labels
                tooltip=tooltip)

    Path(map_output_folder).mkdir(parents=True, exist_ok=True)
    
    filename = Path(map_output_folder) / Path(map_name + '.html')

    r.to_html(filename=filename, open_browser=open_browser, notebook_display=False)

End of functions arrived at from `pydeck_test.ipynb`

Now we want to download the England/UK pop density `.gpkg` file, load it into GeoPandas, format it correctly, and see if first we can plot the population density as we have done before.

I will note that 
* I am currently not super sure whether keeping the data as a GeoDataFrame in GeoPandas (which keeps a geometry column we can do operations on) will be necessary or whether we can use a DataFrame and then just use H3 funcitons on this dataframe
* I think it's probably best to keep in the long run so we can do alternative plotting to pydeck as well as these use the geometry column.
* if pydeck supports using a GeoDataFrame then will do this, just need to check.